# 02 – Utforsk MNIST

**Mål:** Forstå sammenhengen mellom matriser og bilder,
undersøke og visualisere klassefordeling.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn


## Et gråtonebilde er en matrise med tall

Et digitalt bilde består av **piksler**. I et gråtonebilde
beskriver ett tall lysstyrken til hver piksel. Tallene er ordnet i rader
og kolonner: en **matrise**, som vi kan lagre i en todimensjonal tensor.

Her lager vi et bilde med fire rader og fire kolonner. Med skalaen 0–1
betyr 0 svart, 1 hvitt og verdier imellom ulike gråtoner. `image[row, column]`
velger én piksel. Rad 0 ligger øverst, kolonne 0 lengst til venstre.
Fargebilder trenger flere verdier per piksel, vanligvis rødt, grønt og
blått fordelt på 3 kanaler;

Kjør eksemplet. Sammenlign tallmatrisen med figuren: `imshow` oversetter
verdiene til lysstyrke.


In [ ]:
example_image = torch.tensor(
    [
        [0.0, 0.0, 0.0, 0.0],
        [0.0, 0.4, 0.8, 0.0],
        [0.0, 0.8, 1.0, 0.0],
        [0.0, 0.0, 0.0, 0.0],
    ]
)
print("Bildet som tallmatrise:")
print(example_image)
print("Piksel i rad 1, kolonne 2:", example_image[1, 2].item())

fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(example_image, cmap="gray")
for row in range(4):
    for column in range(4):
        value = example_image[row, column].item()
        ax.text(column, row, f"{value:.1f}", ha="center", va="center", color="black" if value > 0.5 else "white")
ax.set_xticks(range(4))
ax.set_yticks(range(4))
ax.set(xlabel="Kolonne", ylabel="Rad", title="De samme tallene vist som et bilde")
plt.show()


## Oppgave 1: Endre én piksel

Hva skjer hvis pikselen i rad 0, kolonne 0 endres
fra 0 til 1? Bruk `clone()` til å lage en uavhengig kopi:

`changed_image = example_image.clone()`

Endre denne pikselen til 1 og vis det nye bildet med
`imshow` og gråtoner.
Skriv også ut den nye matrisen.

<details>
<summary>Hint</summary>

`changed_image[0, 0] = 1.0` endrer øverste venstre piksel.
Bruk figurkoden fra eksemplet.

</details>


In [ ]:
changed_image = example_image.clone()
changed_image[0, 0] = 1.0
print(changed_image)
fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(changed_image, cmap="gray")
ax.set_xticks(range(4))
ax.set_yticks(range(4))
ax.set(xlabel="Kolonne", ylabel="Rad")
plt.show()


**Kontroll:** Kjør cellen når du har endret bildet.


In [ ]:
assert changed_image.shape == example_image.shape
assert changed_image[0, 0].item() == 1.0
assert example_image[0, 0].item() == 0.0
assert (changed_image != example_image).sum().item() == 1


## Ferdig kode: last ned og hent datasettet

Nå skal vi over til å jobbe med faktiske bilder.

Kodesnutten nedenfor laster ned MNIST-datasettet til prosjektets `data/`-mappe.

Et **datasett** lar oss hente ett par `(image, label)` med en indeks,
omtrent som fra en liste. `image` inneholder pikselverdiene, og `label`
forteller hvilket siffer bildet forestiller. Vi undersøker ett bilde
om gangen.


In [ ]:
from pytorch_101.utils import download_mnist

train_data, test_data = download_mnist()
print(f"Treningsbilder: {len(train_data)}, testbilder: {len(test_data)}")


## Samme idé, flere piksler

Rå MNIST-bilder er matriser med 28 rader og 28 kolonner. Hvert tall er
en pikselverdi fra 0 til 255.

Datahjelperen bruker `ToTensor`, som skalerer tallene til 0–1 og legger
til en **kanalakse**. Derfor har bildet formen `[1, 28, 28]`:
én gråtonekanal, 28 rader og 28 kolonner. Klassen lagres separat.
`image[0, row, column]` henter én piksel fra denne ene kanalen.

Cellen under henter råbildet og den skalerte versjonen av samme bilde.
Legg merke til at en liten utskjæring av bildet også bare er tall.


In [ ]:
image, label = train_data[0]
original_index = train_data.indices[0]
raw_image = train_data.dataset.data[original_index]
print("Råbilde:", raw_image.shape, raw_image.dtype, raw_image.min().item(), raw_image.max().item())
print("Etter ToTensor():", image.shape, image.dtype, image.min().item(), image.max().item())
print("Klasse:", label)

print("Et utsnitt av matrisen:")
print(image[0, 10:16, 10:16])


## Oppgave 2: Vis et MNIST-bilde

Trekk en tilfeldig indeks i `train_data`, og hent bilde og etikett derfra.
Kall dem `my_image` og
`my_label`. Vis bildet i gråtoner med etiketten i tittelen, uten akser.
Skriv også ut form, datatype og minste og største pikselverdi.

<details>
<summary>Hint</summary>

`torch.randint(len(train_data), ()).item()` trekker en indeks.
`my_image.squeeze(0)` endrer formen fra `[1, 28, 28]` til `[28, 28]`.

</details>


In [ ]:
image_index = torch.randint(len(train_data), ()).item()
my_image, my_label = train_data[image_index]
fig, ax = plt.subplots(figsize=(3, 3))
ax.imshow(my_image.squeeze(0), cmap="gray", vmin=0, vmax=1)
ax.set_title(f"Riktig siffer: {my_label}")
ax.axis("off")
plt.show()
print(my_image.shape, my_image.dtype)
print("Minste og største verdi:", my_image.min().item(), my_image.max().item())


**Kontroll:** Kjør denne cellen når du har løst oppgaven. En feilmelding betyr at du må undersøke løsningen.


In [ ]:
assert my_image.shape == (1, 28, 28)
assert 0 <= my_label <= 9
assert 0 <= my_image.min() <= my_image.max() <= 1


### Refleksjon

Hva betyr dimensjonene i formen til `my_image`? Hva betyr svart og hvitt med denne fargeskalaen?

Skriv svaret ditt i Markdown-cellen under. Bruk egne resultater der det er relevant.


**Mitt svar:**

Formen er `[1, 28, 28]`: én gråtonekanal, 28 rader og 28 kolonner.
Svart betyr verdi 0 og hvitt betyr verdi 1.


## Eksempel: flere bilder i samme figur

`subplots` lager flere akser. `ravel()` gir oss en flat liste av aksene, slik
at vi kan bruke en vanlig løkke. Hver akse får sitt eget bilde og sin egen tittel.


In [ ]:
example_images = [example_image, 1 - example_image, example_image.T, torch.zeros_like(example_image)]
fig, axes = plt.subplots(2, 2, figsize=(5, 5))
for index, ax in enumerate(axes.ravel()):
    ax.imshow(example_images[index], cmap="gray")
    ax.set_title(f"Eksempel {index}")
    ax.axis("off")
fig.tight_layout()
plt.show()


## Valgfri oppgave 3: Lag et bilderutenett

Lag en figur med 3 × 3 akser. Trekk ni forskjellige treningsbilder og vis dem med riktig
etikett over hvert bilde. Bruk samme gråtoneskala i alle rutene.

<details>
<summary>Hint</summary>

Bruk `torch.randperm(len(train_data))[:9].tolist()` for ni ulike indekser.
Kombiner dem med aksene ved hjelp av `zip(axes.ravel(), image_indices)`.

</details>


In [ ]:
image_indices = torch.randperm(len(train_data))[:9].tolist()
fig, axes = plt.subplots(3, 3, figsize=(6, 6))
for ax, index in zip(axes.ravel(), image_indices):
    image, label = train_data[index]
    ax.imshow(image.squeeze(0), cmap="gray")
    ax.set_title(f"Siffer: {label}")
    ax.axis("off")
fig.tight_layout()
plt.show()


## Eksempel: tell kategorier og tegn et stolpediagram

Klasse-labels er kategorier. Fordelingen kan vises i et stolpediagram.

In [ ]:
fruits = ["eple", "pære", "eple", "banan", "pære", "eple"]
categories = ["eple", "pære", "banan"]
fruit_counts = [fruits.count(category) for category in categories]
fig, ax = plt.subplots()
ax.bar(categories, fruit_counts)
ax.set(xlabel="Frukt", ylabel="Antall", title="Fordeling av frukter")
plt.show()


## Oppgave 4: Undersøk klassefordelingen

Tell hvor mange bilder det er av hvert siffer i **treningsutvalget**.
Lag listen `class_counts` med ti tall, der indeks 0 er antall nuller.
Lag et stolpediagram med siffer på x-aksen og antall bilder på y-aksen.
Vis alle sifrene 0–9 langs x-aksen. Skriv også ut antallene.

<details>
<summary>Hint</summary>

Start med `class_counts = [0] * 10`. For hvert `image, label` i datasettet:
øk `class_counts[label]` med 1. Bruk `range(10)` som stolpenes posisjoner.

</details>

Finn også `majority_class` og `baseline_accuracy`: hvilken nøyaktighet får
en modell som alltid gjetter det vanligste treningssifferet, målt på treningsutvalget?


In [ ]:
class_counts = [0] * 10
for image, label in train_data:
    class_counts[label] += 1
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(10), class_counts)
ax.set(xlabel="Siffer", ylabel="Antall bilder", title="Klassefordeling i treningsutvalget")
ax.set_xticks(range(10))
plt.show()
print("Antall per siffer:", dict(enumerate(class_counts)))
print("Færrest:", min(range(10), key=lambda digit: class_counts[digit]))
print("Flest:", max(range(10), key=lambda digit: class_counts[digit]))

majority_class = max(range(10), key=lambda digit: class_counts[digit])
baseline_accuracy = class_counts[majority_class] / len(train_data)
print(f"Alltid {majority_class}: {baseline_accuracy:.1%} på treningsutvalget")


**Kontroll:** Kjør denne cellen når du har løst oppgaven. En feilmelding betyr at du må undersøke løsningen.


In [ ]:
assert len(class_counts) == 10
assert sum(class_counts) == len(train_data)
assert all(count > 0 for count in class_counts)

assert majority_class == max(range(10), key=lambda digit: class_counts[digit])
assert baseline_accuracy == max(class_counts) / len(train_data)


### Refleksjon

1. Hvilket siffer forekommer oftest og sjeldnest? Oppgi begge antallene fra din kjøring.
2. Er fordelingen omtrent jevn? Hvorfor kan en svært skjev fordeling være et problem
ved trening og når vi rapporterer samlet nøyaktighet?
3. Hvorfor er flertallsgjetting en nyttig referanse?

Skriv svaret ditt i Markdown-cellen under. Bruk egne resultater der det er relevant.



**Mitt svar:**

Med det faste utvalget forekommer sifferet 1 oftest, med 643 bilder, og sifferet
5 sjeldnest, med 553 bilder. Utvalget er omtrent jevnt, men ikke helt balansert.
Tilfeldig utvalg garanterer
ikke like mange av hver klasse. En modell ser færre eksempler fra sjeldne klasser,
og god samlet nøyaktighet kan skjule svak ytelse på disse klassene.

Å alltid gjette 1 gir 643/6000, omtrent 10,7 % på treningsutvalget. En modell
bør lære mer enn dette enkle mønsteret. Treningstall alene viser ikke hvordan
den fungerer på nye bilder; vi trenger en separat evaluering.


## Referanser

- [PyTorch – Datasets & DataLoaders](https://docs.pytorch.org/tutorials/beginner/basics/data_tutorial.html)
